# Benchmarking Quantum Optimization for Microgrid Energy Dispatch

Reproduce the tables and figures of the NAPS 2026 paper from the stored results, in the browser.
No quantum account and no solver run is needed: the notebook reads the per-window results of the
paper runs (IBM Kingston QAOA, D-Wave Advantage2, D-Wave Stride, MILP, exact QUBO enumeration and
classical samplers) and rebuilds every number.

Run the cells from top to bottom.

## 1. Get the repository

In [ ]:
import os, subprocess, sys

REPO = "Benchmarking-Quantum-Optimization-for-Microgrid-Energy-Dispatch"
URL = f"https://github.com/JannatulShorna12/{REPO}.git"

if os.path.exists("scripts/reproduce_paper.py"):
    pass
elif os.path.exists("../scripts/reproduce_paper.py"):
    os.chdir("..")
else:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "--depth", "1", URL], check=True)
    os.chdir(REPO)
print(os.getcwd())

## 2. Rebuild all tables and figures

Writes `results/paper/` and `figures/` and checks every number of the paper against the stored
results.

In [ ]:
run = subprocess.run([sys.executable, "scripts/reproduce_paper.py"], capture_output=True, text=True)
print("\n".join(line for line in run.stdout.splitlines() if not line.startswith("[")))
run.check_returncode()

## 3. Table II: daily operating cost and gap to the MILP

In [ ]:
import pandas as pd
from IPython.display import Image, display

t2 = pd.read_csv("results/paper/table2_daily_cost.csv").set_index("date")
table = pd.DataFrame({
    "MILP": t2.milp_table_column.map("{:.2f}".format),
    "Exact QUBO": [f"{c:.2f} ({d:+.2f})" for c, d in zip(t2.exact_qubo_cost, t2.exact_qubo_delta_pct)],
    "IBM Kingston p=2/3": [f"{c:.2f} ({d:+.2f})" for c, d in zip(t2.ibm_qaoa_p2_p3_cost, t2.ibm_qaoa_p2_p3_delta_pct)],
    "D-Wave QPU": [f"{c:.2f} ({d:+.2f})" for c, d in zip(t2.dwave_qpu_cost, t2.dwave_qpu_delta_pct)],
    "D-Wave Stride": [f"{c:.2f} ({d:+.2f})" for c, d in zip(t2.dwave_stride_cost, t2.dwave_stride_delta_pct)],
})
table

## 4. Fig. 2 and Fig. 3

In [ ]:
display(Image("figures/fig2_daily_cost.png"))
display(Image("figures/fig3_soc_2018-07-23.png"))

## 5. Totals, samplers and the exact QUBO optimum

In [ ]:
pd.read_csv("results/paper/solver_totals.csv").round(2)

## 6. One study day

Pick one of 2018-03-03, 2018-04-13, 2018-04-14, 2018-06-27, 2018-07-23, 2018-10-26.

In [ ]:
date = "2018-04-14"

display(pd.read_csv(f"results/paper/by_date/{date}/summary.csv").round(3))
for name in ("soc", "grid_import", "hourly_cost"):
    display(Image(f"figures/by_date/{date}/{name}.png"))

## 7. Every number of the paper and where it comes from

In [ ]:
numbers = pd.read_csv("results/paper/paper_numbers.csv")
print(numbers.status.value_counts().to_string())
numbers[["section", "quantity", "paper_value", "repository_value", "source"]]

## 8. QUBO benchmark instances

The 144 dispatch-window QUBOs of the paper (20 binary variables each) are in
`results/qubo_instances/`. Each file has the QUBO, the window data and the reference
solution. The cell below loads one instance, checks the reference energy and finds the ground
state by enumerating all 2^20 assignments (a few seconds).

In [ ]:
import json
import numpy as np

inst = json.load(open("results/qubo_instances/2018-07-23/window_17.json"))
n = inst["n"]
h = np.zeros(n)
for i, v in inst["linear"].items():
    h[int(i)] = v
J = np.zeros((n, n))
for i, j, v in inst["quadratic"]:
    J[i, j] = v

def energy(X):
    X = np.atleast_2d(X).astype(float)
    return inst["offset"] + X @ h + np.einsum("ki,ij,kj->k", X, J, X)

x_ref = np.array(inst["reference"]["assignment"])
print("reference energy (file):", inst["reference"]["energy"])
print("reference energy (here):", float(energy(x_ref)[0]))

best_e, best_x = np.inf, None
for lo in range(0, 1 << n, 1 << 16):
    idx = np.arange(lo, lo + (1 << 16))
    X = ((idx[:, None] >> np.arange(n)) & 1)
    e = energy(X)
    k = int(np.argmin(e))
    if e[k] < best_e:
        best_e, best_x = float(e[k]), X[k]
print("ground-state energy (enumeration):", best_e)
print("ground-state energy (file):       ", inst["reference"]["ground_state_energy"])